In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1. Load and Scale
df = pd.read_csv("cleaned_data.csv")
X = df.drop('Outcome', axis=1).values.astype('float32')
y = df['Outcome'].values.astype('float32')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# 2. Define Network
class DiabetesNet(nn.Module):
    def __init__(self):
        super(DiabetesNet, self).__init__()
        self.fc1 = nn.Linear(8, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.sigmoid(self.fc3(x))

model = DiabetesNet()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# 3. Train
inputs = torch.from_numpy(X_train)
labels = torch.from_numpy(y_train).reshape(-1, 1)

for epoch in range(100):
    optimizer.zero_grad()
    loss = criterion(model(inputs), labels)
    loss.backward()
    optimizer.step()

In [ ]:
# 4. Export to ONNX
dummy_input = torch.randn(1, 8)

torch.onnx.export(
    model, 
    dummy_input, 
    "model.onnx",
    input_names=['input'],   # name the input
    output_names=['output'], # name the output
    dynamic_axes={
        'input': {0: 'batch_size'},  # allow the first dimension to be dynamic
        'output': {0: 'batch_size'}
    }
)

In [ ]:
# Save artifacts for next node
pd.DataFrame(X_test).to_csv("X_test_scaled.csv", index=False)
pd.DataFrame(y_test).to_csv("y_test.csv", index=False)
print("PyTorch model exported with dynamic axes for batch inference.")